# Development Silver Validation

Develops and validates the Silver-layer transformation rules using the development sample of the London Smart Meter dataset.

## Purpose

- Standardise source column names
- Cast consumption and timestamp fields to analytical data types
- Apply data-quality validation rules
- Separate rejected records from valid records
- Remove duplicate household-timestamp readings
- Derive analytical date/time attributes
- Reconcile source, rejected, duplicate and Silver row counts

> This notebook validates the Silver transformation logic on the development sample before applying the same rules to the full-scale Bronze dataset.

## 1. Load Development Source

In [36]:
# ----------------------------------------
# Project 05 - Microsoft Fabric Analytics Platform
# Notebook 02 - Silver Transformation
# ----------------------------------------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

source_path = "Files/bronze/meter_readings/*"

print("Silver transformation notebook initialised.")

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 38, Finished, Available, Finished, False)

Silver transformation notebook initialised.


In [37]:
bronze_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(source_path)
)

bronze_count = bronze_df.count()

print("BRONZE INPUT")
print("-" * 50)
print(f"Rows: {bronze_count:,}")
print(f"Columns: {len(bronze_df.columns)}")

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 39, Finished, Available, Finished, False)

BRONZE INPUT
--------------------------------------------------
Rows: 1,000,000
Columns: 4


## 2. Standardise Source Columns

In [38]:
consumption_source_col = "KWH/hh (per half hour) "

standardised_df = (
    bronze_df
    .select(
        F.col("LCLid").alias("HouseholdID"),
        F.col("stdorToU").alias("TariffType"),
        F.col("DateTime").alias("ReadingTimestampRaw"),
        F.col(consumption_source_col).alias("ConsumptionKWhRaw")
    )
)

display(standardised_df.limit(10))

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e3bea060-a4fd-4eef-ad10-6063ee72fd45)

## 3. Apply Data Types and Validation Rules

In [39]:
typed_df = (
    standardised_df
    .withColumn(
        "REadingTimestamp",
        F.to_timestamp("ReadingTimestampRaw")
    )
    .withColumn(
        "ConsumptionKWh",
        F.col("ConsumptionKWhRaw").cast("double")
    )
)

typed_df.printSchema()

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 41, Finished, Available, Finished, False)

root
 |-- HouseholdID: string (nullable = true)
 |-- TariffType: string (nullable = true)
 |-- ReadingTimestampRaw: string (nullable = true)
 |-- ConsumptionKWhRaw: string (nullable = true)
 |-- REadingTimestamp: timestamp (nullable = true)
 |-- ConsumptionKWh: double (nullable = true)



## 4. Separate Valid and Rejected Records

In [40]:
validated_df = (
    typed_df
    .withColumn(
        "IsMissingHousehold",
        F.col("HouseholdID").isNull() |
        (F.trim(F.col("HouseholdID")) == "")
    )
    .withColumn(
        "IsInvalidTimestamp",
        F.col("ReadingTimestamp").isNull()
    )
    .withColumn(
        "IsInvalidConsumption",
        F.col("ConsumptionKWh").isNull() |
        (F.col("ConsumptionKWh") < 0)
    )
    .withColumn(
        "IsInvalidTariff",
        F.col("TariffType").isNull() |
        (~F.col("TariffType").isin("Std", "ToU"))
    )
)

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 42, Finished, Available, Finished, False)

In [41]:
validated_df = (
    validated_df
    .withColumn(
        "RejectionReason",
        F.when(
            F.col("IsMissingHousehold"),
            F.lit("MISSING_HOUSEHOLD")
        )
        .when(
            F.col("IsInvalidTimestamp"),
            F.lit("INVALID_TIMESTAMP")
        )
        .when(
            F.col("IsInvalidConsumption"),
            F.lit("INVALID_CONSUMPTION")
        )
        .when(
            F.col("IsInvalidTariff"),
            F.lit("INVALID_TARIFF")
        )
        .otherwise(F.lit(None))
    )
)

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 43, Finished, Available, Finished, False)

In [42]:
rejected_df = (
    validated_df
    .filter(F.col("RejectionReason").isNotNull())
)

valid_df = (
    validated_df
    .filter(F.col("RejectionReason").isNull())
)

rejected_count = rejected_df.count()
valid_before_dedup_count = valid_df.count()

print("VALIDATION RESULTS")
print("-" * 50)
print(f"Bronze rows: {bronze_count:,}")
print(f"Valid before deduplication: {valid_before_dedup_count:,}")
print(f"Rejected rows: {rejected_count:,}")

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 44, Finished, Available, Finished, False)

VALIDATION RESULTS
--------------------------------------------------
Bronze rows: 1,000,000
Valid before deduplication: 999,971
Rejected rows: 29


In [43]:
print("REJECTION SUMMARY")
print("-" * 50)

rejected_df.groupBy("RejectionReason").count().orderBy(
    F.desc("count")
).show(truncate=False)

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 45, Finished, Available, Finished, False)

REJECTION SUMMARY
--------------------------------------------------
+-------------------+-----+
|RejectionReason    |count|
+-------------------+-----+
|INVALID_CONSUMPTION|29   |
+-------------------+-----+



## 5. Deduplicate Business Keys

In [44]:
dedup_window = (
    Window
    .partitionBy("HouseholdID", "ReadingTimestamp")
    .orderBy(
        F.col("ConsumptionKWh").asc_nulls_last(),
        F.col("TariffType").asc_nulls_last()
    )
)

ranked_df = (
    valid_df
    .withColumn(
        "DuplicateRank",
        F.row_number().over(dedup_window)
    )
)

duplicates_removed_df = (
    ranked_df
    .filter(F.col("DuplicateRank") > 1)
)

silver_base_df = (
    ranked_df
    .filter(F.col("DuplicateRank") == 1)
)

duplicates_removed_count = duplicates_removed_df.count()

print("DEDUPLICATION")
print("-" * 50)
print(f"Duplicate rows removed: {duplicates_removed_count:,}")

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 46, Finished, Available, Finished, False)

DEDUPLICATION
--------------------------------------------------
Duplicate rows removed: 688


## 6. Derive Analytical Attributes

In [45]:
silver_df = (
    silver_base_df
    .withColumn(
        "ReadingDate",
        F.to_date("ReadingTimestamp")
    )
    .withColumn(
        "ReadingYear",
        F.year("ReadingTimestamp")
    )
    .withColumn(
        "ReadingMonth",
        F.month("ReadingTimestamp")
    )
    .withColumn(
        "ReadingDay",
        F.dayofmonth("ReadingTimestamp")
    )
    .withColumn(
        "ReadingHour",
        F.hour("ReadingTimestamp")
    )
    .withColumn(
        "DayOfWeek",
        F.date_format("ReadingTimestamp", "EEEE")
    )
)

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 47, Finished, Available, Finished, False)

In [46]:

silver_final_df = (
    silver_df
    .select(
        "HouseholdID",
        "TariffType",
        "ReadingTimestamp",
        "ConsumptionKWh",
        "ReadingDate",
        "ReadingYear",
        "ReadingMonth",
        "ReadingDay",
        "ReadingHour",
        "DayOfWeek"
    )
)

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 48, Finished, Available, Finished, False)

## 7. Reconciliation and Quality Checks

In [47]:
silver_count = silver_final_df.count()

print("SILVER RECONCILIATION")
print("-" * 50)
print(f"Bronze rows:               {bronze_count:,}")
print(f"Rejected rows:             {rejected_count:,}")
print(f"Duplicates removed:        {duplicates_removed_count:,}")
print(f"Silver rows:               {silver_count:,}")
print(
    f"Reconciliation difference: "
    f"{bronze_count - rejected_count - duplicates_removed_count - silver_count:,}"
)

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 49, Finished, Available, Finished, False)

SILVER RECONCILIATION
--------------------------------------------------
Bronze rows:               1,000,000
Rejected rows:             29
Duplicates removed:        688
Silver rows:               999,283
Reconciliation difference: 0


In [48]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

print("Silver schema ready.")


StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 50, Finished, Available, Finished, False)

Silver schema ready.


In [49]:

(
    silver_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.meter_readings")
)

print("silver.meter_readings written successfully.")

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 51, Finished, Available, Finished, False)

silver.meter_readings written successfully.


In [50]:
rejected_output_df = (
    rejected_df
    .select(
        "HouseholdID",
        "TariffType",
        "ReadingTimestampRaw",
        "ConsumptionKWhRaw",
        "RejectionReason"
    )
)

(
    rejected_output_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.rejected_meter_readings")
)

print("silver.rejected_meter_readings written successfully.")

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 52, Finished, Available, Finished, False)

silver.rejected_meter_readings written successfully.


## 8. Reconciliation and Quality Checks

In [51]:
silver_check_df = spark.table("silver.meter_readings")
rejected_check_df = spark.table("silver.rejected_meter_readings")

print("PERSISTED SILVER TABLES")
print("-" * 50)
print(f"silver.meter_readings: {silver_check_df.count():,}")
print(f"silver.rejected_meter_readings: {rejected_check_df.count():,}")

display(silver_check_df.limit(10))

StatementMeta(, dad532d0-dfd3-414d-b25f-2e41bae3464f, 53, Finished, Available, Finished, False)

PERSISTED SILVER TABLES
--------------------------------------------------
silver.meter_readings: 999,283
silver.rejected_meter_readings: 29


SynapseWidget(Synapse.DataFrame, 20284d4e-b18d-4e42-b896-e48525ca57ca)